## Import necessary libraries

In [1]:
# general libraries
import numpy as np

# for conversion of stochastic matrix to unitary matrix
from scipy.linalg import null_space

## Dagger Operation

In [2]:
def dagger(A):
    return np.conj(A).T

## Conversion from Stochastic Matrix to Unitary Matrix

In [3]:
def stochastic_to_unitary(A, verbose=False):
    """
    Convert an s x n column-stochastic matrix A into an ns x ns block
    diagonal unitary matrix U, following Algorithm 1.

    Each column c_i of A is used to build an s x s unitary block U_i:
    sqrt(c_i) (entrywise) is used as the first column (valid because
    columns of a column-stochastic matrix sum to 1, so sqrt(c_i) has
    unit norm), and the remaining s-1 columns are an orthonormal basis
    for the null space of the matrix whose rows are all sqrt(c_i)
    (equivalently, vectors orthogonal to sqrt(c_i)). The n resulting
    s x s unitary blocks U_1, ..., U_n are placed along the diagonal
    of an ns x ns matrix, with zeros elsewhere.

    Parameters
    ----------
    A : ndarray of shape (s, n)
        Column-stochastic matrix: every column is non-negative and
        sums to 1 (so each column can be treated as a probability
        distribution over s outcomes).
    verbose : bool, optional (default=False)
        If True, prints each block U_i as it's constructed (along with
        its shape and null space dimension), a sanity check that each
        U_i is unitary (U_i^T @ U_i should be ~I_s), the final shape
        of U, and a sanity check that U itself is unitary
        (U^T @ U should be ~I_ns).

    Returns
    -------
    U : ndarray of shape (n*s, n*s)
        Block diagonal unitary matrix with n blocks of size s x s
        along the diagonal (one per column of A), zeros elsewhere.
    """
    U = []
    for i in range(A.shape[1]):
        s_s = np.array([np.sqrt(A[:, i])] * A.shape[0])
        null_s_s = null_space(s_s)
        s_s = np.hstack([np.sqrt(A[:, i:i+1]), null_s_s])

        if verbose:
            print(f'Column {i+1} -> unitary block U_{i+1}:')
            print(s_s)
            print(f'  shape: {s_s.shape}, null space dim: {null_s_s.shape[1]}')
            print(f'  U_{i+1}^T @ U_{i+1} (should be ~I):')
            print(np.round(s_s.T @ s_s, 6))
            print()

        zero_m = np.zeros(s_s.shape)
        U_row_i = []
        for j in range(A.shape[1]):
            if j == 0:
                U_row_i = s_s if i == j else zero_m
            else:
                U_row_i = np.hstack([U_row_i, s_s if i == j else zero_m])
        if i == 0:
            U = U_row_i
        else:
            U = np.vstack([U, U_row_i])

    if verbose:
        print(f'Final block-diagonal U shape: {U.shape}')
        print(f'U^T @ U (should be ~I_{A.shape[0]*A.shape[1]}):')
        print(np.round(U.T @ U, 6))
        print()

    return U

## Tensor Product as Matrix Operation

In [4]:
def tensor_product_W(n, s, verbose=False):
    """
    Construct the ns x n isometry matrix W used to implement a tensor
    product with a fixed s x s environment density matrix rho_B, via
    conjugation: rho_A (tensor) rho_B = W @ rho_A @ W.T

    Here rho_B is the fixed s x s matrix with a single 1 in the top-left
    corner (rho_B[0, 0] = 1) and zeros everywhere else. W is built by
    stacking n blocks of shape (s, n) vertically, where the i-th block
    has a 1 at position (0, i) and zeros elsewhere.

    Parameters
    ----------
    n : int
        Dimension of the density matrix rho_A (n x n) to be tensored,
        and the number of columns in W.
    s : int
        Dimension of the fixed environment density matrix rho_B (s x s),
        and the row-block size in W (each block is s x n).
    verbose : bool, optional (default=False)
        If True, prints each block as it's constructed, the final
        shape of W, and a sanity check (W^T @ W, which should equal
        the n x n identity matrix since W has orthonormal columns).

    Returns
    -------
    W : ndarray of shape (s * n, n)
        The constructed isometry matrix, satisfying
        W @ rho_A @ W.T == np.kron(rho_A, rho_B)
        for any n x n matrix rho_A.
    """
    W = np.array([])
    for i in range(n):
        zeros_i = np.zeros((s, n))
        zeros_i[0, i] = 1

        if verbose:
            print(f'Block {i+1}/{n} (1 placed at row 0, col {i}):')
            print(zeros_i)
            print()

        if i == 0:
            W = zeros_i
        else:
            W = np.vstack([W, zeros_i])

    if verbose:
        print(f'Final W shape: {W.shape} (expected ({s*n}, {n}))')
        print('W:')
        print(W)
        print()
        print('W^T @ W (should be ~I_n, confirms W has orthonormal columns):')
        print(np.round(W.T @ W, 6))
        print()

    return W

## Partial Trace as Matrix Operation

Partial Trace after Projection

In [5]:
def partial_trace_after_projection_V(n, s, y, verbose=False):
    """
    Construct the n x ns matrix V_y used to compute the partial trace
    over an s-dimensional subsystem AFTER a projection has been applied,
    via conjugation:

        tr_B( P_y @ rho_AB @ P_y^dagger ) = V_y @ (P_y @ rho_AB @ P_y^dagger) @ V_y^dagger

    where rho_AB is the joint density matrix of an n-dimensional system
    A and an s-dimensional system B, and P_y is a projection operator
    corresponding to observing outcome y on system B.

    V_y is built from n blocks of shape (n, s), where the i-th block
    has a single 1 at position (i, y) and zeros elsewhere. The blocks
    are concatenated horizontally to form the final (n, n*s) matrix.

    Parameters
    ----------
    n : int
        Dimension of the subsystem A that remains after tracing out B
        (i.e. the number of row-blocks / the output dimension of V_y).
    s : int
        Dimension of the subsystem B being traced out (i.e. the width
        of each block, and the column index space that y indexes into).
    y : int
        Zero-indexed observed outcome on subsystem B (corresponds to
        the "yth column" in the paper's one-indexed notation, so the
        paper's y=2 corresponds to calling this function with y=1).
    verbose : bool, optional (default=False)
        If True, prints each block as it's constructed, and the final
        shape and contents of V_y.

    Returns
    -------
    V : ndarray of shape (n, n * s)
        The constructed matrix V_y.
    """
    V = []
    for i in range(n):
        zeros_i = np.zeros((n, s))
        zeros_i[i][y] = 1

        if verbose:
            print(f'Block {i+1}/{n} (1 placed at row {i}, col {y}):')
            print(zeros_i)
            print()

        if i == 0:
            V = zeros_i
        else:
            V = np.hstack([V, zeros_i])

    if verbose:
        print(f'Final V shape: {V.shape} (expected ({n}, {n*s}))')
        print('V:')
        print(V)

    return V

Partial Trace without Projection

In [6]:
def partial_trace_V(n, s, w, verbose=False):
    """
    Construct a single s x ns matrix V_w that traces out the SYSTEM
    (control) index, keeping the ENVIRONMENT/ancilla register, for a
    specific value of w. Used (summed over all w) to compute the
    unconditional/no-projection partial trace:

        tr_A(rho_AB) = sum_w V_w @ rho_AB @ V_w^T

    V_w is built from n blocks of shape s x s: the w-th block is the
    s x s identity, all other blocks are s x s zeros. The blocks are
    concatenated horizontally.

    Parameters
    ----------
    n : int
        Dimension of the subsystem being traced out (system/control).
    s : int
        Dimension of the subsystem being kept (environment/ancilla).
    w : int
        Which of the n control blocks gets the identity (0-indexed).
    verbose : bool, optional (default=False)
        If True, prints each block and the final matrix.

    Returns
    -------
    V : ndarray of shape (s, n * s)
    """
    V = []
    for i in range(n):
        mat_w = np.identity(s) if i == w else np.zeros((s, s))

        if verbose:
            print(f'Block {i+1}/{n} (identity if i==w={w}):')
            print(mat_w)
            print()

        if i == 0:
            V = mat_w
        else:
            V = np.hstack([V, mat_w])

    if verbose:
        print(f'Final V_{w} shape: {V.shape} (expected ({s}, {n*s}))')
        print(V)

    return V

## Projection Operator

In [7]:
def projection_operator_P(n, s, y):
    """
    Construct P_y = I_n ⊗ |y><y|_s, the ns x ns projector that
    projects the s-dimensional environment/ancilla register onto
    basis state y, leaving the n-dimensional system register
    untouched.

    Parameters
    ----------
    n : int
        Dimension of the system register (left factor).
    s : int
        Dimension of the environment/ancilla register (right factor).
    y : int
        Which basis state (0-indexed) of the environment to project onto.

    Returns
    -------
    P_y : ndarray of shape (n*s, n*s)
        Idempotent projector (P_y @ P_y == P_y), with trace n.
    """
    e_y = np.array([[1] if i == y else [0] for i in range(s)])
    proj_s = e_y @ e_y.T   # |y><y|, s x s
    return np.kron(np.eye(n), proj_s)

## Generate the set of Kraus Operators

Generating the Kraus operators for U1

In [8]:
def generate_Kraus_w_set(n, U1):
    K = []
    W = tensor_product_W(n, n) # tensor p_{t-1} with p_X_t wherein both have same size
    for i in range(n):
        V_w = partial_trace_V(n, n, i) # partial trace of p_{t-1} (nxn)
        K_w = V_w @ U1 @ W
        K.append(K_w)
    return K

Generating the Kraus operators for U2

In [9]:
def generate_Kraus_y_set(n, s, U2):
    K = []
    W = tensor_product_W(n, s) # tensor p_X_t (nxn) with p_Y_t (sxs)
    for i in range(s):
        P_y = projection_operator_P(n, s, i)
        V_y = partial_trace_after_projection_V(n, s, i) # partial trace of p_Y_t (sxs)
        K_w = V_y @ U2 @ W
        K.append(K_w)
    return K

Generating the combined K_y and K_w

In [33]:
def generate_Kraus_w_y_set(A, B):
    n = A.shape[0]
    s = B.shape[0]
    
    U1 = stochastic_to_unitary(A)
    U2 = stochastic_to_unitary(B)
    
    K_w_set = generate_Kraus_w_set(n, U1)
    K_y_set = generate_Kraus_y_set(n, s, U2)

    K = {}
    for i in range(n):
        for j in range(s):
            K[i, j] = K_y_set[j] @ K_w_set[i]

    return K

## Sample

In [34]:
A = np.array([
#FR  R    C    S
    [0.5, 0.4, 0.0], # R
    [0.3, 0.2, 0.3], # C
    [0.2, 0.4, 0.7]  # S
                     # TO
])
B = np.array([
#FR  R    C    S
    [0.8, 0.4, 0.1], # :)
    [0.2, 0.6, 0.9]  # :(
                     # Observed
])

Verify the property $ \sum_i K^\dagger K = I $

In [40]:
n = A.shape[0]
s = B.shape[0]

K = generate_Kraus_w_y_set(A, B)

sum_i_j = []
for i in range(n):
    sum_j = []
    for j in range(s):
        if j == 0:
            sum_j = dagger(K[i, j]) @ K[i, j]
        else:
            sum_j += dagger(K[i, j]) @ K[i, j]
    print(sum_j)
    print()
    if i == 0:
        sum_i_j = sum_j.copy()
    else:
        sum_i_j += sum_j

print("FINAL")
print(sum_i_j)

[[1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

[[0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 1.]]

FINAL
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


This proves that the generated matrices are Kraus operators.